# Plant Leaves SR - Fast ESRGAN-lite cGAN v16 TopK

This notebook uses a fast ESRGAN-lite conditional GAN with AMP, EMA-smoothed generator weights, top-k checkpoint averaging, and 8-way self-ensemble inference.

It writes `/kaggle/working/submission_fast_esrgan_lite_v16_topk.csv` but does not submit anything.

In [1]:
import csv
import math
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models


In [2]:
SEED = 42
SCALE = 4
PATCH_SIZE = 32
BATCH_SIZE = 8
EPOCHS = 45
TRAIN_PATCHES_PER_IMAGE = 16
WARMUP_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 0.0008

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU: {gpu_name} (sm_{major}{minor})')
    DEVICE = torch.device('cuda') if major >= 7 else torch.device('cpu')
else:
    DEVICE = torch.device('cpu')

USE_AMP = DEVICE.type == 'cuda'
print('Using device:', DEVICE)
print('AMP enabled:', USE_AMP)

GPU: Tesla T4 (sm_75)
Using device: cuda
AMP enabled: True


In [3]:
input_root = Path('/kaggle/input')
candidate_lr_dirs = sorted(input_root.rglob('train_Low_Resolution'))
assert candidate_lr_dirs, 'Could not find competition data.'

TRAIN_LR_DIR = candidate_lr_dirs[0]
DATASET_ROOT = TRAIN_LR_DIR.parent
TRAIN_HR_DIR = DATASET_ROOT / 'train_High_Resolution'
TEST_LR_DIR = DATASET_ROOT / 'test_Low_Resolution'
SAMPLE_PATH = DATASET_ROOT / 'sample_submission.csv'
VGG_PATH = DATASET_ROOT / 'vgg19_weights.pth'
OUT_DIR = Path('/kaggle/working')
SUBMISSION_PATH = OUT_DIR / 'submission_fast_esrgan_lite_v16_topk.csv'
CKPT_PATH = OUT_DIR / 'best_fast_esrgan_lite_v16_topk.pt'

print('Dataset root:', DATASET_ROOT)
print('Has VGG weights:', VGG_PATH.exists())

Dataset root: /kaggle/input/competitions/plant-leaves-super-resolution-challenge
Has VGG weights: True


In [4]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0

def to_tensor(arr):
    return torch.from_numpy(arr).permute(2, 0, 1)

def to_uint8(tensor):
    arr = tensor.detach().clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return np.clip(np.round(arr * 255.0), 0, 255).astype(np.uint8)

def image_mae(pred, target):
    return float(np.mean(np.abs(pred.astype(np.float32) - target.astype(np.float32))))

all_lr_paths = sorted(TRAIN_LR_DIR.glob('*.png'))
split_idx = int(len(all_lr_paths) * 0.9)
train_lr_paths = all_lr_paths[:split_idx]
val_lr_paths = all_lr_paths[split_idx:]

print('train_images:', len(train_lr_paths))
print('val_images:', len(val_lr_paths))

train_images: 1477
val_images: 165


In [5]:
class TrainDataset(Dataset):
    def __init__(self, lr_paths, patch_size=32, patches_per_image=16):
        self.lr_paths = list(lr_paths)
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image

    def __len__(self):
        return len(self.lr_paths) * self.patches_per_image

    def __getitem__(self, idx):
        lr_path = self.lr_paths[idx % len(self.lr_paths)]
        hr_path = TRAIN_HR_DIR / lr_path.name
        lr = load_rgb(lr_path)
        hr = load_rgb(hr_path)

        h, w = lr.shape[:2]
        ps = min(self.patch_size, h, w)
        top = random.randint(0, h - ps)
        left = random.randint(0, w - ps)

        lr_patch = lr[top:top+ps, left:left+ps]
        hr_patch = hr[top*SCALE:(top+ps)*SCALE, left*SCALE:(left+ps)*SCALE]

        if random.random() < 0.5:
            lr_patch = np.flip(lr_patch, axis=1).copy()
            hr_patch = np.flip(hr_patch, axis=1).copy()
        if random.random() < 0.5:
            lr_patch = np.flip(lr_patch, axis=0).copy()
            hr_patch = np.flip(hr_patch, axis=0).copy()
        k = random.randint(0, 3)
        if k:
            lr_patch = np.rot90(lr_patch, k).copy()
            hr_patch = np.rot90(hr_patch, k).copy()

        bicubic = np.asarray(
            Image.fromarray(np.clip(np.round(lr_patch * 255.0), 0, 255).astype(np.uint8)).resize((ps * SCALE, ps * SCALE), Image.Resampling.BICUBIC),
            dtype=np.float32
        ) / 255.0
        residual = hr_patch - bicubic

        return to_tensor(lr_patch), to_tensor(bicubic), to_tensor(hr_patch), to_tensor(residual)


In [6]:
class ResidualDenseBlock(nn.Module):
    def __init__(self, channels=64, growth=24):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, growth, 3, padding=1)
        self.conv2 = nn.Conv2d(channels + growth, growth, 3, padding=1)
        self.conv3 = nn.Conv2d(channels + growth * 2, growth, 3, padding=1)
        self.conv4 = nn.Conv2d(channels + growth * 3, growth, 3, padding=1)
        self.conv5 = nn.Conv2d(channels + growth * 4, channels, 3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.act(self.conv1(x))
        x2 = self.act(self.conv2(torch.cat([x, x1], dim=1)))
        x3 = self.act(self.conv3(torch.cat([x, x1, x2], dim=1)))
        x4 = self.act(self.conv4(torch.cat([x, x1, x2, x3], dim=1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], dim=1))
        return x + 0.2 * x5

class RRDB(nn.Module):
    def __init__(self, channels=64, growth=24):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(channels, growth)
        self.rdb2 = ResidualDenseBlock(channels, growth)
        self.rdb3 = ResidualDenseBlock(channels, growth)

    def forward(self, x):
        return x + 0.2 * self.rdb3(self.rdb2(self.rdb1(x)))

class Generator(nn.Module):
    def __init__(self, channels=64, num_rrdb=4, growth=24, scale=4):
        super().__init__()
        self.head = nn.Conv2d(3, channels, 3, padding=1)
        self.body = nn.Sequential(*[RRDB(channels, growth) for _ in range(num_rrdb)])
        self.body_conv = nn.Conv2d(channels, channels, 3, padding=1)
        up_layers = []
        for _ in range(int(math.log2(scale))):
            up_layers += [
                nn.Conv2d(channels, channels * 4, 3, padding=1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.2, inplace=True)
            ]
        self.up = nn.Sequential(*up_layers)
        self.tail = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, 3, 3, padding=1),
        )

    def forward(self, x, bicubic_hr):
        feat = self.head(x)
        body = self.body_conv(self.body(feat)) + feat
        residual = self.tail(self.up(body))
        return bicubic_hr + residual

class Discriminator(nn.Module):
    def __init__(self, in_channels=6, base=24):
        super().__init__()
        def block(cin, cout, stride=1, norm=True):
            layers = [nn.Conv2d(cin, cout, 3, stride=stride, padding=1)]
            if norm:
                layers.append(nn.InstanceNorm2d(cout))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.net = nn.Sequential(
            *block(in_channels, base, norm=False),
            *block(base, base, stride=2),
            *block(base, base * 2),
            *block(base * 2, base * 2, stride=2),
            *block(base * 2, base * 4),
            *block(base * 4, base * 4, stride=2),
            nn.Conv2d(base * 4, 1, 3, padding=1)
        )

    def forward(self, cond_hr, target_hr):
        x = torch.cat([cond_hr, target_hr], dim=1)
        return self.net(x)

class VGGPerceptualLoss(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        vgg = models.vgg19(weights=None)
        state_dict = torch.load(weights_path, map_location='cpu')
        vgg.load_state_dict(state_dict)
        self.features = nn.Sequential(*list(vgg.features.children())[:18]).eval()
        for p in self.features.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred_n = (pred - self.mean) / self.std
        target_n = (target - self.mean) / self.std
        return F.l1_loss(self.features(pred_n), self.features(target_n))

def fft_loss(pred, target):
    pred_fft = torch.fft.rfft2(pred, norm='ortho')
    target_fft = torch.fft.rfft2(target, norm='ortho')
    return F.l1_loss(torch.abs(pred_fft), torch.abs(target_fft))


In [7]:
train_ds = TrainDataset(train_lr_paths, patch_size=PATCH_SIZE, patches_per_image=TRAIN_PATCHES_PER_IMAGE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=DEVICE.type == 'cuda', drop_last=True)

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
def charbonnier_loss(pred, target, eps=1e-3):
    return torch.mean(torch.sqrt((pred - target) ** 2 + eps ** 2))

perc_loss = VGGPerceptualLoss(VGG_PATH).to(DEVICE)
adv_loss = nn.BCEWithLogitsLoss()

opt_g = torch.optim.AdamW(G.parameters(), lr=1.5e-4, betas=(0.9, 0.99), weight_decay=1e-4)
opt_d = torch.optim.AdamW(D.parameters(), lr=3e-5, betas=(0.9, 0.99), weight_decay=1e-4)
scaler_g = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_d = torch.cuda.amp.GradScaler(enabled=USE_AMP)
EMA_DECAY = 0.999
EMA_EVERY = 1
TOP_K = 3
ema_state = {k: v.detach().clone() for k, v in G.state_dict().items()}
best_ckpts = []

def update_ema(model, ema_state, decay):
    with torch.no_grad():
        msd = model.state_dict()
        for k, v in msd.items():
            ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)

def load_ema(model, ema_state):
    model.load_state_dict({k: v.clone() for k, v in ema_state.items()})

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(max(1, WARMUP_EPOCHS))
    progress = float(epoch - WARMUP_EPOCHS) / float(max(1, EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

sch_g = torch.optim.lr_scheduler.LambdaLR(opt_g, lr_lambda=lr_lambda)
sch_d = torch.optim.lr_scheduler.LambdaLR(opt_d, lr_lambda=lr_lambda)

def evaluate(generator, lr_paths):
    generator.eval()
    scores = []
    with torch.no_grad():
        for lr_path in lr_paths:
            hr_path = TRAIN_HR_DIR / lr_path.name
            lr_np = load_rgb(lr_path)
            bicubic_np = np.asarray(Image.open(lr_path).convert('RGB').resize((128, 128), Image.Resampling.BICUBIC), dtype=np.float32) / 255.0
            lr = to_tensor(lr_np).unsqueeze(0).to(DEVICE)
            bicubic = to_tensor(bicubic_np).unsqueeze(0).to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred = generator(lr, bicubic).squeeze(0)
            pred_u8 = to_uint8(pred)
            target_u8 = np.asarray(Image.open(hr_path).convert('RGB'), dtype=np.uint8)
            scores.append(image_mae(pred_u8, target_u8))
    return float(np.mean(scores))

def bicubic_score(lr_paths):
    scores = []
    for lr_path in lr_paths:
        hr_path = TRAIN_HR_DIR / lr_path.name
        pred = Image.open(lr_path).convert('RGB').resize((128, 128), Image.Resampling.BICUBIC)
        pred_u8 = np.asarray(pred, dtype=np.uint8)
        target_u8 = np.asarray(Image.open(hr_path).convert('RGB'), dtype=np.uint8)
        scores.append(image_mae(pred_u8, target_u8))
    return float(np.mean(scores))

print('val_bicubic_mae:', bicubic_score(val_lr_paths))

/tmp/ipykernel_23/2818008637.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_g = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_23/2818008637.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_d = torch.cuda.amp.GradScaler(enabled=USE_AMP)


val_bicubic_mae: 18.19605763464263


In [8]:
best_val = float('inf')
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    G.train()
    D.train()
    g_losses = []
    d_losses = []
    for step_idx, (lr_patch, bicubic_hr, hr_patch, residual_target) in enumerate(train_loader, start=1):
        lr_patch = lr_patch.to(DEVICE, non_blocking=True)
        bicubic_hr = bicubic_hr.to(DEVICE, non_blocking=True)
        hr_patch = hr_patch.to(DEVICE, non_blocking=True)
        residual_target = residual_target.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake_hr = G(lr_patch, bicubic_hr)

        if epoch > WARMUP_EPOCHS:
            opt_d.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                real_logits = D(bicubic_hr, hr_patch)
                fake_logits = D(bicubic_hr, fake_hr.detach())
                d_real = adv_loss(real_logits, torch.ones_like(real_logits))
                d_fake = adv_loss(fake_logits, torch.zeros_like(fake_logits))
                d_loss = 0.5 * (d_real + d_fake)
            scaler_d.scale(d_loss).backward()
            scaler_d.unscale_(opt_d)
            torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
            scaler_d.step(opt_d)
            scaler_d.update()
            d_losses.append(d_loss.item())

        opt_g.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            fake_hr = G(lr_patch, bicubic_hr)
            residual_pred = fake_hr - bicubic_hr
            l_pix = charbonnier_loss(fake_hr, hr_patch)
            l_res = charbonnier_loss(residual_pred, residual_target)
            l_perc = perc_loss(fake_hr, hr_patch)
            l_freq = fft_loss(fake_hr, hr_patch)
            if epoch > WARMUP_EPOCHS:
                logits = D(bicubic_hr, fake_hr)
                l_adv = adv_loss(logits, torch.ones_like(logits))
            else:
                l_adv = torch.tensor(0.0, device=DEVICE)

            g_loss = 1.15 * l_pix + 0.55 * l_res + 0.012 * l_perc + 0.008 * l_freq + 0.00005 * l_adv
        scaler_g.scale(g_loss).backward()
        scaler_g.unscale_(opt_g)
        torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
        scaler_g.step(opt_g)
        scaler_g.update()
        if step_idx % EMA_EVERY == 0 if 'step_idx' in locals() else True:
            update_ema(G, ema_state, EMA_DECAY)
        g_losses.append(g_loss.item())

    sch_g.step()
    if epoch > WARMUP_EPOCHS:
        sch_d.step()
    raw_state = {k: v.detach().clone() for k, v in G.state_dict().items()}
    load_ema(G, ema_state)
    val_mae = evaluate(G, val_lr_paths)
    G.load_state_dict(raw_state)
    mean_g = float(np.mean(g_losses))
    mean_d = float(np.mean(d_losses)) if d_losses else 0.0
    lr_g = opt_g.param_groups[0]['lr']
    print(f'Epoch {epoch}/{EPOCHS} - lr_g: {lr_g:.7f} - g_loss: {mean_g:.6f} - d_loss: {mean_d:.6f} - val_mae: {val_mae:.6f}')

    if val_mae < best_val - EARLY_STOPPING_MIN_DELTA:
        best_val = val_mae
        epochs_without_improvement = 0
        raw_state = {k: v.detach().clone() for k, v in G.state_dict().items()}
        load_ema(G, ema_state)
        payload = {'G': G.state_dict(), 'D': D.state_dict(), 'val_mae': best_val}
        torch.save(payload, CKPT_PATH)
        ckpt_copy = {k: (v.detach().clone() if torch.is_tensor(v) else v) for k, v in payload['G'].items()}
        best_ckpts.append((best_val, ckpt_copy))
        best_ckpts = sorted(best_ckpts, key=lambda x: x[0])[:TOP_K]
        G.load_state_dict(raw_state)
        print('Saved best checkpoint:', CKPT_PATH)
    else:
        epochs_without_improvement += 1
        print(f'No significant improvement for {epochs_without_improvement} epoch(s).')

    if epoch > WARMUP_EPOCHS and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print('Early stopping triggered.')
        break


/tmp/ipykernel_23/1248129196.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_23/1248129196.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_23/2818008637.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 1/45 - lr_g: 0.0000300 - g_loss: 0.133445 - d_loss: 0.000000 - val_mae: 17.598216
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 2/45 - lr_g: 0.0000450 - g_loss: 0.131125 - d_loss: 0.000000 - val_mae: 17.426988
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 3/45 - lr_g: 0.0000600 - g_loss: 0.130132 - d_loss: 0.000000 - val_mae: 17.330757
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 4/45 - lr_g: 0.0000750 - g_loss: 0.129351 - d_loss: 0.000000 - val_mae: 17.250188
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 5/45 - lr_g: 0.0000900 - g_loss: 0.128692 - d_loss: 0.000000 - val_mae: 17.181961
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 6/45 - lr_g: 0.0001050 - g_loss: 0.128071 - d_loss: 0.000000 - val_mae: 17.116163
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 7/45 - lr_g: 0.0001200 - g_l

/tmp/ipykernel_23/1248129196.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 11/45 - lr_g: 0.0001497 - g_loss: 0.126316 - d_loss: 0.476589 - val_mae: 16.971431
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 12/45 - lr_g: 0.0001488 - g_loss: 0.126157 - d_loss: 0.153918 - val_mae: 16.955564
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 13/45 - lr_g: 0.0001473 - g_loss: 0.126040 - d_loss: 0.036840 - val_mae: 16.942745
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 14/45 - lr_g: 0.0001452 - g_loss: 0.125983 - d_loss: 0.008665 - val_mae: 16.931272
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 15/45 - lr_g: 0.0001426 - g_loss: 0.125946 - d_loss: 0.002648 - val_mae: 16.923007
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 16/45 - lr_g: 0.0001394 - g_loss: 0.125899 - d_loss: 0.001961 - val_mae: 16.914059
Saved best checkpoint: /kaggle/working/best_fast_esrgan_lite_v16_topk.pt
Epoch 17/45 - lr_g: 0.000135

In [9]:
best_ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
G.load_state_dict(best_ckpt['G'])
G.eval()

def infer_with_tta(generator, lr_tensor, bicubic_tensor):
    variants = [
        (lr_tensor, bicubic_tensor, lambda x: x),
        (torch.flip(lr_tensor, dims=[3]), torch.flip(bicubic_tensor, dims=[3]), lambda x: torch.flip(x, dims=[3])),
        (torch.flip(lr_tensor, dims=[2]), torch.flip(bicubic_tensor, dims=[2]), lambda x: torch.flip(x, dims=[2])),
        (torch.flip(lr_tensor, dims=[2, 3]), torch.flip(bicubic_tensor, dims=[2, 3]), lambda x: torch.flip(x, dims=[2, 3])),
        (torch.rot90(lr_tensor, 1, dims=[2, 3]), torch.rot90(bicubic_tensor, 1, dims=[2, 3]), lambda x: torch.rot90(x, -1, dims=[2, 3])),
        (torch.rot90(lr_tensor, 2, dims=[2, 3]), torch.rot90(bicubic_tensor, 2, dims=[2, 3]), lambda x: torch.rot90(x, -2, dims=[2, 3])),
        (torch.rot90(lr_tensor, 3, dims=[2, 3]), torch.rot90(bicubic_tensor, 3, dims=[2, 3]), lambda x: torch.rot90(x, -3, dims=[2, 3])),
        (torch.flip(torch.rot90(lr_tensor, 1, dims=[2, 3]), dims=[3]), torch.flip(torch.rot90(bicubic_tensor, 1, dims=[2, 3]), dims=[3]), lambda x: torch.rot90(torch.flip(x, dims=[3]), -1, dims=[2, 3])),
    ]
    preds = []
    with torch.no_grad():
        for lr_inp, bicubic_inp, inv in variants:
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                preds.append(inv(generator(lr_inp, bicubic_inp)))
    return torch.stack(preds, dim=0).mean(dim=0)

def infer_with_checkpoint_ensemble(generator, checkpoint_states, lr_tensor, bicubic_tensor):
    preds = []
    raw_state = {k: v.detach().clone() for k, v in generator.state_dict().items()}
    with torch.no_grad():
        for _, state_dict in checkpoint_states:
            generator.load_state_dict(state_dict, strict=True)
            preds.append(infer_with_tta(generator, lr_tensor, bicubic_tensor))
    generator.load_state_dict(raw_state, strict=True)
    return torch.stack(preds, dim=0).mean(dim=0)

ensemble_states = best_ckpts if best_ckpts else [(best_ckpt['val_mae'], best_ckpt['G'])]

csv.field_size_limit(10**9)
with SAMPLE_PATH.open('r', newline='') as fin, SUBMISSION_PATH.open('w', newline='') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=['Id', 'Pixels'])
    writer.writeheader()

    for row in reader:
        image_id = row['Id']
        lr_path = TEST_LR_DIR / image_id
        lr_np = load_rgb(lr_path)
        bicubic_np = np.asarray(Image.open(lr_path).convert('RGB').resize((128, 128), Image.Resampling.BICUBIC), dtype=np.float32) / 255.0
        lr = to_tensor(lr_np).unsqueeze(0).to(DEVICE)
        bicubic = to_tensor(bicubic_np).unsqueeze(0).to(DEVICE)
        pred = infer_with_checkpoint_ensemble(G, ensemble_states, lr, bicubic).squeeze(0)
        pred_u8 = to_uint8(pred)
        pixels = ' '.join(map(str, pred_u8.reshape(-1).tolist()))
        writer.writerow({'Id': image_id, 'Pixels': pixels})

print('Wrote submission file:', SUBMISSION_PATH)
print('Best validation MAE:', best_ckpt['val_mae'])

/tmp/ipykernel_23/901385944.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Wrote submission file: /kaggle/working/submission_fast_esrgan_lite_v16_topk.csv
Best validation MAE: 16.88721431385387
